# Path B Pilot 补 seed: STATE-ONLY 三分类 seeds [1, 2]

**目的**：补 seed 1, 2 两个 run，与已存在的 seed 0 合并算 3-seed mean ± std，验证 state-only vs multitask 的 Δ 是否稳定。

| Field | Value |
|---|---|
| Sweep | `configs/sweeps/pathB_cma_state_only_pilot_seeds12.yaml` |
| Task | `cma_pathB_state_only`（与 seed 0 同名，目录并列） |
| Setting | V+K+T / cross_subject / seeds [1, 2]（**2 runs**） |
| runs_root | `runs/pathB_state_only_pilot/`（与 seed 0 共用） |

**已知 seed 0 结果**：
- state-only test_macro_f1_state = 0.4061
- multitask  test_macro_f1_state = 0.3917 (3-seed mean = 0.3924, std = 0.001)
- Δ (single seed) = **+0.0144**（踩 0.015 阈值）

**决策表（Cell 4 自动输出，基于 3-seed mean）**：

| \|Δ F1_state\| (3-seed mean) | 决策 |
|---|---|
| < 0.012 | 用现有 multitask 数据，0 重跑 |
| 0.012–0.04 | 重跑 path B + CMA + EFT-CDF state-only (~72 runs) |
| > 0.04 | 全套重跑 (~264 runs) |

In [ ]:
# Cell 1: Mount Drive + setup
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyyaml', 'scipy', '-q'])

%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2: Dry run — verify plan = exactly 2 runs (seed 1, seed 2)
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/pathB_cma_state_only_pilot_seeds12.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/pathB_state_only_pilot \
    --dry_run 2>&1 | tail -10

In [ ]:
# Cell 3: Run seed 1 + seed 2 (re-run safe — skips seed 0 if already done)
# -u: unbuffered stdout for real-time per-epoch progress
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/pathB_cma_state_only_pilot_seeds12.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/pathB_state_only_pilot

In [ ]:
# Cell 4: 3-seed aggregation — state-only vs multitask, with decision rule
import json, glob
import numpy as np

SO_PATTERN = ('/content/drive/MyDrive/AmuCS_experiment/runs/pathB_state_only_pilot/'
              'cross_subject/*/triple_video_km_event_telem_60hz/*seed{}*/metrics.json')
MT_PATTERN = ('/content/drive/MyDrive/AmuCS_experiment/runs/pathB_cma_state_trend/'
              'cross_subject/cma_pathB_state_trend_3seed/triple_video_km_event_telem_60hz/'
              '*seed{}*/metrics.json')

def load_three(pattern, label):
    f1, bacc = [], []
    for s in [0, 1, 2]:
        files = sorted(glob.glob(pattern.format(s)))
        assert files, f'{label} seed{s} missing: {pattern.format(s)}'
        m = json.load(open(files[-1]))  # newest if multiple
        f1.append(m.get('test_macro_f1_state', m.get('test_macro_f1_mean')))
        bacc.append(m.get('test_balanced_acc_state', m.get('test_balanced_acc_mean')))
    return f1, bacc

so_f1, so_bacc = load_three(SO_PATTERN, 'state-only')
mt_f1, mt_bacc = load_three(MT_PATTERN, 'multitask')

so_f1_m, so_f1_s = float(np.mean(so_f1)), float(np.std(so_f1))
mt_f1_m, mt_f1_s = float(np.mean(mt_f1)), float(np.std(mt_f1))
so_b_m,  so_b_s  = float(np.mean(so_bacc)), float(np.std(so_bacc))
mt_b_m,  mt_b_s  = float(np.mean(mt_bacc)), float(np.std(mt_bacc))

delta_f1 = so_f1_m - mt_f1_m
delta_b  = so_b_m  - mt_b_m

print('='*78)
print('STATE-ONLY 3-seed  vs  MULTITASK 3-seed  (cross_subject / V+K+T)')
print('='*78)
print(f'  test_macro_f1_state    state-only={so_f1_m:.4f}±{so_f1_s:.4f}  multitask={mt_f1_m:.4f}±{mt_f1_s:.4f}  Δ={delta_f1:+.4f}')
print(f'  test_balanced_acc_state state-only={so_b_m:.4f}±{so_b_s:.4f}  multitask={mt_b_m:.4f}±{mt_b_s:.4f}  Δ={delta_b:+.4f}')
print()
print(f'  Per-seed F1_state  state-only = {[f"{x:.4f}" for x in so_f1]}')
print(f'  Per-seed F1_state  multitask  = {[f"{x:.4f}" for x in mt_f1]}')

# Effect-size relative to multitask noise
if mt_f1_s > 1e-6:
    print(f'\n  Δ / multitask_std = {delta_f1 / mt_f1_s:+.2f}× (sanity check vs noise)')

print('\nDECISION:')
abs_delta = abs(delta_f1)
if abs_delta < 0.012:
    print(f'  Δ={delta_f1:+.4f} < 0.012  →  USE EXISTING multitask data, 0 re-runs needed')
elif abs_delta < 0.04:
    print(f'  0.012 ≤ Δ={delta_f1:+.4f} < 0.04  →  RE-RUN path B + CMA + EFT-CDF state-only (~72 runs)')
else:
    print(f'  Δ={delta_f1:+.4f} ≥ 0.04  →  FULL re-run all baselines state-only (~264 runs)')